# Отчёт по этапу 4 дипломного проекта

**Тема:** «Разработка интеллектуального Telegram-бота на основе языковой модели для консультирования по нормативной документации в области электроэнергетики (ПУЭ, ПТЭЭП, ГОСТ)» (проект Diplom_AI — RAG-система по нормативным документам).

**Этап 4:** создание прототипа и получение первой точности (для RAG: прототип рабочего контура и метрики Hit@k, MRR/nDCG, генерации и цитирования).

**Студент:** Aleksander Ozyaz  
**Группа:** ____________________  
**Руководитель:** ____________________  
**Репозиторий:** https://github.com/kirag-ozyaz/Diplom_AI

---

**Как пользоваться.** Нажмите *Run All* (Выполнить все). Ноутбук читает только сохранённые файлы результатов из `scripts/stage4_eval/results/` (и `data/`); Milvus, Ollama и GPU не нужны. Корень репозитория определяется автоматически. Повторный прогон оценки (нужны Docker, Milvus, Ollama) — в разделе 6, по умолчанию выключен флагом `RUN_LIVE = False`.

Графики интерактивные (Plotly): наведите курсор, чтобы увидеть значение, щёлкните по легенде, чтобы скрыть серию, выделите участок, чтобы приблизить. Числа те же, что в отчёте Word; документ Word этим ноутбуком не формируется.

Опыты пронумерованы с 1: опыт 1 — база (`diplom_multimodal`), опыт 2 — префиксы E5 (`diplom_multimodal_e5`), опыт 3 — E5 + реранкер (`diplom_multimodal_e5_rerank`). В файлах проекта те же опыты обозначены 0, 1, 2 (суффиксы JSON: нет / `_e5` / `_rerank`).


## 0. Подготовка: поиск корня проекта и вспомогательные функции

In [1]:
# Флаг повторного прогона оценки (раздел 6). False — только чтение сохранённых результатов.
RUN_LIVE = False

import json, re
from collections import Counter
from pathlib import Path
import plotly.graph_objects as go
import numpy as np
from IPython.display import Markdown, display

def find_root(start=None):
    """Корень репозитория: первая папка вверх от текущей, где есть scripts/stage4_eval/results."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "stage4_eval" / "results").is_dir():
            return p
    raise FileNotFoundError("Не найден корень проекта (нет scripts/stage4_eval/results). "
                            "Откройте ноутбук внутри репозитория Diplom_AI.")

ROOT = find_root()
RES = ROOT / "scripts" / "stage4_eval" / "results"
RDIR, GDIR = RES / "retrieval", RES / "generation"
REPORTS = ROOT / "Этапы" / "Reports" / "etap4"
QUESTIONS = ROOT / "data" / "rag_eval_questions" / "eval_questions.json"

EXPERIMENTS = [  # (номер, подпись, суффикс hitk/gen/perf/judge, режим rank)
    (1, "Опыт 1 (база)", "", "base"),
    (2, "Опыт 2 (E5)", "_e5", "e5"),
    (3, "Опыт 3 (реранкер)", "_rerank", "rerank"),
]
LABELS = [e[1] for e in EXPERIMENTS]
COLORS = ["#7f7f7f", "#1f77b4", "#2ca02c"]

def load(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8-sig")) if p.is_file() else None

def num(v, d=1):
    """Число в русском формате: 81.3 -> '81,3'."""
    if v is None:
        return "—"
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return f"{v:.{d}f}".replace(".", ",")
    return str(v)

def md_table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

def _show(fig, title, ylabel, ymax):
    """Общее оформление живого графика Plotly."""
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=460,
        margin=dict(t=88, b=48, l=56, r=24),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        hovermode="x unified",
    )
    axis = dict(title=ylabel)
    if ymax is not None:
        axis["range"] = [0, ymax]
    fig.update_yaxes(**axis)
    fig.show()

def bar_chart(categories, series, title, ylabel, ymax=None, fmt="{:.1f}", colors=COLORS):
    """Сгруппированная столбчатая диаграмма: series = [(подпись, [значения])]."""
    fig = go.Figure()
    xs = [str(c) for c in categories]
    for i, (name, vals) in enumerate(series):
        y, text = [], []
        for v in vals:
            if v is None:
                y.append(None)
                text.append("")
            else:
                y.append(float(v))
                text.append(fmt.format(v).replace(".", ","))
        fig.add_trace(go.Bar(
            name=name, x=xs, y=y, text=text, textposition="outside", cliponaxis=False,
            marker_color=colors[i % len(colors)],
            hovertemplate="%{x}<br>%{fullData.name}: %{text}<extra></extra>",
        ))
    fig.update_layout(barmode="group")
    _show(fig, title, ylabel, ymax)

def line_chart(categories, series, title, ylabel, ymax=None, colors=None):
    """Линии с маркерами: series = [(подпись, [значения])]."""
    palette = colors or ["#1f77b4", "#ff7f0e", "#2ca02c"]
    fig = go.Figure()
    xs = [str(c) for c in categories]
    for i, (name, vals) in enumerate(series):
        fig.add_trace(go.Scatter(
            name=name, x=xs,
            y=[None if v is None else float(v) for v in vals],
            mode="lines+markers",
            line=dict(color=palette[i % len(palette)], width=2),
            marker=dict(size=8),
            hovertemplate="%{x}<br>%{fullData.name}: %{y}<extra></extra>",
        ))
    _show(fig, title, ylabel, ymax)

# Загрузка всех результатов трёх опытов
DATA = {}
for n, label, suf, mode in EXPERIMENTS:
    DATA[n] = {
        "hitk": load(RDIR / f"stage4_hitk_results{suf}.json"),
        "rank": load(RDIR / f"stage4_rank_results_{mode}.json"),
        "gen": load(GDIR / f"stage4_gen_eval_results{suf}.json"),
        "perf": load(GDIR / f"stage4_performance_metrics{suf}.json"),
        "judge": load(GDIR / f"stage4_llm_judge_metrics{suf}.json"),
    }
print("Корень проекта:", ROOT)
for n, label, suf, mode in EXPERIMENTS:
    have = [k for k, v in DATA[n].items() if v is not None]
    miss = [k for k, v in DATA[n].items() if v is None]
    print(f"{label}: есть {have}" + (f"; нет {miss}" if miss else ""))


Корень проекта: X:\Учеба_УИИ\Итоговы_Проект
Опыт 1 (база): есть ['hitk', 'rank', 'gen', 'perf', 'judge']
Опыт 2 (E5): есть ['hitk', 'rank', 'gen', 'perf', 'judge']
Опыт 3 (реранкер): есть ['hitk', 'rank', 'gen', 'perf', 'judge']


## 1. Тема, описание задачи

**Задача проекта.** Пользователь задаёт вопрос на русском языке, система находит подходящие пункты нормативного документа и формирует ответ со ссылками на номера пунктов. Используется схема RAG: семантический поиск фрагментов + генерация ответа языковой моделью по найденным фрагментам.

**Задача этапа 4** (по рекомендации куратора объём сужен до ПУЭ):

1. собрать прототип рабочего контура «вопрос → эмбеддинг → Milvus → top-k фрагментов → LLM → ответ со ссылками на пункты ПУЭ»;
2. подготовить контрольный набор вопросов с эталонными пунктами ПУЭ;
3. получить первую точность: Hit@1/3/5 для поиска, MRR и nDCG@k, метрики генерации и цитирования;
4. провести опыты по улучшению поиска и сравнить их на одном наборе вопросов.

Модели предобученные, обучение весов не выполнялось, поэтому графиков loss/accuracy по эпохам нет. «Первой точностью» считается Hit@k — доля вопросов, для которых эталонный пункт ПУЭ попал в первые k найденных фрагментов.

## 2. База данных

**Корпус:** ПУЭ, 7-е издание, главы в DOCX (`data/raw/Нормативная база/ПУЭ/DOCX/`) → Markdown (`data/extracted/`) → чанки JSONL (`data/chunked/`). Документ открытый, конфиденциальных данных нет. ПТЭ (приказ Минэнерго № 811), ПТЭЭП и ГОСТ на этапе 4 в индекс не включались.

Ячейка ниже считает объём базы и наборов вопросов по файлам репозитория.

In [3]:
chunk_dir = ROOT / "data" / "chunked"
files = sorted(chunk_dir.glob("*.chunked.jsonl")) if chunk_dir.is_dir() else []
if files:
    recs = [json.loads(l) for f in files for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]
    clauses = [r["metadata"].get("Clause") for r in recs if r["metadata"].get("Clause")]
    md_table(["Показатель", "Значение"], [
        ["Файлов DOCX (ПУЭ)", len(list((ROOT / "data/raw/Нормативная база/ПУЭ/DOCX").glob("*.docx")))],
        ["Файлов Markdown", len(list((ROOT / "data/extracted").glob("*.md")))],
        ["Файлов чанков JSONL", len(files)],
        ["Чанков всего", len(recs)],
        ["Чанков с номером пункта (Clause)", f"{len(clauses)} (различных: {len(set(clauses))})"],
        ["Чанков без номера пункта", len(recs) - len(clauses)],
        ["Чанков с таблицами", sum(1 for r in recs if r["metadata"].get("contains_tables"))],
        ["Чанков с изображениями", sum(1 for r in recs if r["metadata"].get("contains_images"))],
    ])
else:
    print("Папка data/chunked не найдена — статистика базы пропущена.")

rows = []
for f in sorted((ROOT / "data" / "rag_eval_questions").glob("*.json")):
    d = load(f)
    n = d.get("n_questions", len(d.get("questions", []))) if isinstance(d, dict) else len(d)
    rows.append([f"`{f.name}`", n, (d.get("registry_range", "") if isinstance(d, dict) else "")])
md_table(["Набор вопросов", "Вопросов", "Диапазон registry_id"], rows)

qs = load(QUESTIONS)["questions"]
SECTION = {q["id"]: q["section"].split(".")[0].replace("Раздел ", "Р.") for q in qs}
cnt = Counter(SECTION.values())
print(f"Итоговый рабочий набор eval_questions.json: {len(qs)} вопросов; по разделам ПУЭ:",
      ", ".join(f"{k} — {cnt[k]}" for k in sorted(cnt)))

| Показатель | Значение |
|---|---|
| Файлов DOCX (ПУЭ) | 44 |
| Файлов Markdown | 44 |
| Файлов чанков JSONL | 43 |
| Чанков всего | 2912 |
| Чанков с номером пункта (Clause) | 2838 (различных: 2838) |
| Чанков без номера пункта | 74 |
| Чанков с таблицами | 165 |
| Чанков с изображениями | 184 |

| Набор вопросов | Вопросов | Диапазон registry_id |
|---|---|---|
| `eval_questions.json` | 268 | Q-0560 – Q-0827 |
| `eval_questions_300_final.json` | 268 | Q-0560 – Q-0827 |
| `eval_questions_374_final.json` | 337 | Q-0828 – Q-1164 |
| `eval_questions_74.json` | 74 |  |
| `eval_questions_base38.json` | 38 |  |
| `eval_questions_pue300_no_test24_final.json` | 259 | Q-0301 – Q-0559 |
| `eval_questions_pue_all.json` | 1164 |  |
| `eval_questions_pue_expanded_final.json` | 559 | Q-0001 – Q-0559 |

Итоговый рабочий набор eval_questions.json: 268 вопросов; по разделам ПУЭ: Р.1 — 42, Р.2 — 45, Р.3 — 35, Р.4 — 41, Р.5 — 39, Р.6 — 31, Р.7 — 35


## 3. Параметризация данных

**Сегментация.** `src/preprocessing/Create_chunkeds/md_to_chunked_2.py`: уровни Document → Section → Chapter → Paragraph → Clause; один пункт ПУЭ — один чанк; перед текстом добавляется строка контекста «документ. раздел. глава. параграф. Пункт X.Y.Z».

**Индекс Milvus** (по паспортам прогонов): HNSW по `text_vector`, COSINE, M=8, efConstruction=200, поиск ef=64, top-k = 5. Для MRR/nDCG — выдача глубиной 10.

Параметры трёх опытов из конфигурационных файлов:

In [4]:
cfg_paths = {
    1: ROOT / "config" / "rag_runtime.json",
    2: ROOT / "src/preprocessing/Create_embeddings/E5/rag_runtime_e5.json",
    3: ROOT / "src/preprocessing/Create_embeddings/Rerank/rag_runtime_rerank.json",
}
cfg = {k: load(p) for k, p in cfg_paths.items()}
def g(c, *path):
    for p in path:
        c = (c or {}).get(p)
    return c
rows = []
for name, getter in [
    ("Конфиг", lambda n: f"`{cfg_paths[n].relative_to(ROOT).as_posix()}`"),
    ("Коллекция Milvus", lambda n: g(cfg[n], "vector_db", "collection_name")),
    ("Модель эмбеддингов", lambda n: g(cfg[n], "models", "text_model_name")),
    ("Устройство", lambda n: g(cfg[n], "models", "device_text")),
    ("Префиксы query:/passage:", lambda n: "да" if n > 1 else "нет"),
    ("Реранкер", lambda n: g(cfg[n], "rerank", "model") or "нет"),
    ("Кандидатов из Milvus", lambda n: g(cfg[n], "rerank", "candidates") or g(cfg[n], "query", "default_limit")),
    ("Фрагментов в LLM (top-k)", lambda n: g(cfg[n], "query", "default_limit")),
    ("Записей в коллекции (по JSON Hit@k)", lambda n: g(DATA[n]["hitk"], "collection_num_entities")),
    ("LLM (Ollama)", lambda n: g(cfg[n], "ollama", "model")),
    ("temperature / num_predict", lambda n: f'{g(cfg[n], "ollama", "temperature")} / {g(cfg[n], "ollama", "num_predict")}'),
]:
    rows.append([name] + [getter(n) for n in (1, 2, 3)])
md_table(["Параметр"] + LABELS, rows)

| Параметр | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| Конфиг | `config/rag_runtime.json` | `src/preprocessing/Create_embeddings/E5/rag_runtime_e5.json` | `src/preprocessing/Create_embeddings/Rerank/rag_runtime_rerank.json` |
| Коллекция Milvus | diplom_multimodal | diplom_multimodal_e5 | diplom_multimodal_e5_rerank |
| Модель эмбеддингов | intfloat/multilingual-e5-base | intfloat/multilingual-e5-base | intfloat/multilingual-e5-base |
| Устройство | cuda | cuda | cuda |
| Префиксы query:/passage: | нет | да | да |
| Реранкер | нет | нет | BAAI/bge-reranker-v2-m3 |
| Кандидатов из Milvus | 5 | 5 | 20 |
| Фрагментов в LLM (top-k) | 5 | 5 | 5 |
| Записей в коллекции (по JSON Hit@k) | 2912 | 2912 | 2912 |
| LLM (Ollama) | qwen2.5:14b-instruct-q4_K_M | qwen2.5:14b-instruct-q4_K_M | qwen2.5:14b-instruct-q4_K_M |
| temperature / num_predict | 0.2 / 512 | 0.2 / 512 | 0.2 / 512 |

## 4. Архитектура рабочего контура (прототип RAG)

```
Вопрос пользователя
   │  ("query: " + вопрос — в опытах 2 и 3)
   ▼
multilingual-e5-base → вектор 768
   ▼
Milvus (HNSW, COSINE) → top-5 (опыты 1–2) / top-20 (опыт 3)
   ▼
[опыт 3] bge-reranker-v2-m3 → top-5
   ▼
Промпт: системная инструкция + 5 фрагментов ПУЭ + вопрос
   ▼
Ollama qwen2.5:14b → ответ с номерами пунктов ПУЭ
```

| Компонент | Файл |
|---|---|
| Подготовка чанков | `src/preprocessing/Create_chunkeds/md_to_chunked_2.py` |
| Класс RAG (эмбеддинги, Milvus, реранкер) | `src/preprocessing/Create_embeddings/multimodal_rag.py` |
| Загрузка в Milvus, опыты 1 / 2 / 3 | `load_data.py`, `E5/load_data_e5.py`, `Rerank/load_data_rerank.py` |
| Сервис RAG (поиск + ответ) | `src/rag/rag_service.py`, `src/rag/ollama_client.py` |
| Конфигурация | `config/rag_runtime.json` и конфиги опытов 2–3 |
| Инфраструктура | `infra/milvus/`, `infra/docker/Dockerfile.ollama/` |

Telegram-бот (`src/bot/`) к контуру на этапе 4 не подключён.

## 5. Графическое подтверждение (метрики прототипа)

- **Hit@k** (k = 1, 3, 5) — доля вопросов, где в первых k фрагментах есть номер эталонного пункта (`\b{clause}\b`, «Пункт {clause}», «п. {clause}»); один поиск, без LLM.
- **MRR, nDCG@k** — по позиции первого фрагмента с эталонным пунктом в выдаче глубиной 10; при одном релевантном пункте nDCG@k = 1/log2(rank+1).
- **Генерация (live)** — citation_rate, citation_in_answer_rate, groundedness, relevance, время запроса.
- **Performance** — нагрузочный тест Ollama (без Milvus и реранкера).
- **LLM-as-a-Judge** — оценка той же моделью по шаблонному контексту (вспомогательная).

### 5.1. Опыт 1 (база): серия прогонов

Для опыта 1 проведена серия прогонов на последовательных версиях набора вопросов (25, 38, 50, 73, 300, итоговый 268). Источник — журнал `stage4_hitk_runs.jsonl`; подряд идущие прогоны на одном N объединены.

In [5]:
runs = [json.loads(l) for l in (RDIR / "stage4_hitk_runs.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
series = []  # группы подряд идущих прогонов с одинаковым N
for r in runs:
    if series and series[-1]["n"] == r["n_questions"]:
        series[-1]["runs"].append(r)
    else:
        series.append({"n": r["n_questions"], "runs": [r]})
rows = []
for i, s in enumerate(series, 1):
    first, last = s["runs"][0], s["runs"][-1]
    m = first["metrics_percent"]
    same = all(x["metrics_percent"] == m for x in s["runs"])
    rows.append([f"1.{i}", s["n"], first["evaluated_at"] + ("" if len(s["runs"]) == 1 else f" … {last['evaluated_at']}"),
                 len(s["runs"]), num(m["hit@1"]) + " %", num(m["hit@3"]) + " %", num(m["hit@5"]) + " %",
                 "да" if same else "нет"])
md_table(["№", "N", "Дата (первый … последний)", "Прогонов", "Hit@1", "Hit@3", "Hit@5", "Повторы совпали"], rows)

line_chart(
    [f"N={s['n']}" for s in series],
    [(k.replace("hit", "Hit"), [s["runs"][0]["metrics_percent"][k] for s in series])
     for k in ("hit@1", "hit@3", "hit@5")],
    "Рисунок 1 — Опыт 1: Hit@k в серии прогонов", "%", ymax=100,
)


| № | N | Дата (первый … последний) | Прогонов | Hit@1 | Hit@3 | Hit@5 | Повторы совпали |
|---|---|---|---|---|---|---|---|
| 1.1 | 25 | 29.09.2026 00:36 … 07.10.2026 01:36 | 12 | 40,0 % | 44,0 % | 48,0 % | да |
| 1.2 | 38 | 07.10.2026 22:11 … 08.10.2026 00:48 | 3 | 55,3 % | 63,2 % | 68,4 % | да |
| 1.3 | 50 | 08.10.2026 01:03 | 1 | 46,0 % | 54,0 % | 58,0 % | да |
| 1.4 | 73 | 08.10.2026 01:10 | 1 | 68,5 % | 76,7 % | 79,5 % | да |
| 1.5 | 300 | 18.10.2026 12:15 | 1 | 12,3 % | 23,0 % | 28,0 % | да |
| 1.6 | 268 | 18.10.2026 16:58 … 25.10.2026 20:00 | 2 | 58,2 % | 76,1 % | 81,3 % | да |

Падение на 300 вопросах вызвано составом набора (в том числе автоматической разметкой эталонных пунктов), а не поиском. После проверки набор сокращён до 268 вопросов; повторный прогон на 268 вопросах дал те же значения.

### 5.2. Опыт 2 (префиксы E5)

Префиксы `passage:` при индексации и `query:` при поиске, отдельная коллекция `diplom_multimodal_e5`.

### 5.3. Опыт 3 (E5 + реранкер)

Префиксы как в опыте 2; Milvus возвращает 20 кандидатов, `BAAI/bge-reranker-v2-m3` оставляет 5 лучших (для MRR/nDCG — 10).

### 5.4. Сравнение опытов: Hit@k

In [6]:
rows = []
for k in ["hit@1", "hit@3", "hit@5"]:
    rows.append([k.replace("hit", "Hit")] + [
        f'{num(DATA[n]["hitk"]["metrics_percent"][k])} % ({sum(d["hit"]["@" + k[4:]] for d in DATA[n]["hitk"]["details"])}/{DATA[n]["hitk"]["n_questions"]})'
        for n in (1, 2, 3)])
MISS5 = {n: {d["id"] for d in DATA[n]["hitk"]["details"] if not d["hit"]["@5"]} for n in (1, 2, 3)}
rows.append(["Промахов Hit@5"] + [len(MISS5[n]) for n in (1, 2, 3)])
rows.append(["Дата прогона"] + [DATA[n]["hitk"]["evaluated_at"] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["Hit@1", "Hit@3", "Hit@5"],
          [(LABELS[n - 1], [DATA[n]["hitk"]["metrics_percent"][k] for k in ("hit@1", "hit@3", "hit@5")]) for n in (1, 2, 3)],
          "Рисунок 2 — Hit@k по трём опытам (N = 268)", "%", ymax=100)
print(f"Опыт 2 исправил {len(MISS5[1] - MISS5[2])} промахов Hit@5 опыта 1 и добавил {len(MISS5[2] - MISS5[1])}; "
      f"опыт 3 исправил {len(MISS5[2] - MISS5[3])} промахов опыта 2 и добавил {len(MISS5[3] - MISS5[2])}; "
      f"не найдено ни в одном опыте: {len(MISS5[1] & MISS5[2] & MISS5[3])}.")

| Метрика | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| Hit@1 | 58,2 % (156/268) | 60,1 % (161/268) | 72,0 % (193/268) |
| Hit@3 | 76,1 % (204/268) | 78,0 % (209/268) | 85,4 % (229/268) |
| Hit@5 | 81,3 % (218/268) | 83,6 % (224/268) | 88,8 % (238/268) |
| Промахов Hit@5 | 50 | 44 | 30 |
| Дата прогона | 25.10.2026 20:00 | 25.10.2026 21:04 | 26.10.2026 00:40 |

Опыт 2 исправил 9 промахов Hit@5 опыта 1 и добавил 3; опыт 3 исправил 19 промахов опыта 2 и добавил 5; не найдено ни в одном опыте: 23.


Hit@5 по разделам ПУЭ:

In [7]:
secs = sorted(set(SECTION.values()))
tab = {}
for n in (1, 2, 3):
    tot, hit = Counter(), Counter()
    for d in DATA[n]["hitk"]["details"]:
        s = SECTION[d["id"]]; tot[s] += 1; hit[s] += d["hit"]["@5"]
    tab[n] = {s: 100 * hit[s] / tot[s] for s in secs}
md_table(["Раздел", "Вопросов"] + LABELS, [[s, cnt[s]] + [num(tab[n][s]) + " %" for n in (1, 2, 3)] for s in secs])
bar_chart(secs, [(LABELS[n - 1], [tab[n][s] for s in secs]) for n in (1, 2, 3)],
          "Рисунок 3 — Hit@5 по разделам ПУЭ", "%", ymax=110, fmt="{:.0f}")

| Раздел | Вопросов | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|---|
| Р.1 | 42 | 78,6 % | 81,0 % | 88,1 % |
| Р.2 | 45 | 66,7 % | 73,3 % | 82,2 % |
| Р.3 | 35 | 82,9 % | 85,7 % | 91,4 % |
| Р.4 | 41 | 78,0 % | 78,0 % | 85,4 % |
| Р.5 | 39 | 97,4 % | 97,4 % | 92,3 % |
| Р.6 | 31 | 93,5 % | 93,5 % | 96,8 % |
| Р.7 | 35 | 77,1 % | 80,0 % | 88,6 % |

### 5.5. Сравнение опытов: MRR и nDCG@k (глубина 10)

In [8]:
RANK_KEYS = ["hit@10", "mrr", "mrr@3", "mrr@5", "ndcg@3", "ndcg@5", "ndcg@10"]
RNAME = {"hit@10": "Hit@10", "mrr": "MRR", "mrr@3": "MRR@3", "mrr@5": "MRR@5", "ndcg@3": "nDCG@3", "ndcg@5": "nDCG@5", "ndcg@10": "nDCG@10"}
if all(DATA[n]["rank"] for n in (1, 2, 3)):
    rows = [[RNAME[k]] + [num(DATA[n]["rank"]["metrics"][k], 3) for n in (1, 2, 3)] for k in RANK_KEYS]
    rows.append(["Дата прогона"] + [DATA[n]["rank"]["evaluated_at"] for n in (1, 2, 3)])
    md_table(["Метрика"] + LABELS, rows)
    for n in (1, 2, 3):  # проверка: Hit@1/3/5 в rank-прогоне = основной прогон
        for k in ("hit@1", "hit@3", "hit@5"):
            a, b = DATA[n]["hitk"]["metrics_percent"][k], round(DATA[n]["rank"]["metrics"][k] * 100, 1)
            if abs(a - b) > 0.05:
                print(f"! {LABELS[n-1]}: {k} {a} ≠ {b}")
    bar_chart([RNAME[k] for k in RANK_KEYS], [(LABELS[n - 1], [DATA[n]["rank"]["metrics"][k] for k in RANK_KEYS]) for n in (1, 2, 3)],
              "Рисунок 4 — MRR и nDCG@k по опытам", "0–1", ymax=1.05, fmt="{:.2f}")
else:
    print("Нет stage4_rank_results_*.json — запустите run_rank_all.ps1 (раздел 6).")

| Метрика | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| Hit@10 | 0,877 | 0,888 | 0,929 |
| MRR | 0,681 | 0,696 | 0,793 |
| MRR@3 | 0,661 | 0,677 | 0,779 |
| MRR@5 | 0,673 | 0,689 | 0,787 |
| nDCG@3 | 0,687 | 0,703 | 0,799 |
| nDCG@5 | 0,708 | 0,726 | 0,812 |
| nDCG@10 | 0,729 | 0,743 | 0,826 |
| Дата прогона | 28.10.2026 00:54 | 28.10.2026 00:54 | 28.10.2026 01:00 |

### 5.6. Сравнение опытов: генерация и цитирование

Числа генерации берутся только из JSON. Если файл посчитан не на всех вопросах (`partial_run` или N ≠ 268), его значения в таблицу и графики не попадают.

In [9]:
GEN_KEYS = ["citation_rate", "citation_in_answer_rate", "citation_in_context_rate", "groundedness_avg", "relevance_avg"]

GEN, GEN_SRC = {}, {}
for n, label, suf, mode in EXPERIMENTS:
    g_ = DATA[n]["gen"]; N = DATA[n]["hitk"]["n_questions"]
    m = (g_ or {}).get("metrics", {})
    if g_ is not None and not m.get("partial_run") and m.get("n_questions") == N:
        GEN[n] = dict(m); GEN[n]["errors"] = sum(1 for d in g_["details"] if d.get("error"))
        GEN_SRC[n] = f"stage4_gen_eval_results{suf}.json ({g_['evaluated_at']})"
    else:
        GEN[n] = {}
        GEN_SRC[n] = "нет полного прогона"
rows = [[k] + [num(GEN[n].get(k), 1 if k.endswith("rate") else 3) for n in (1, 2, 3)] for k in GEN_KEYS]
rows.append(["Среднее время запроса, с"] + [num(GEN[n].get("duration_avg_sec"), 2) for n in (1, 2, 3)])
rows.append(["Ошибок live"] + [GEN[n].get("errors", "нет данных") for n in (1, 2, 3)])
rows.append(["Источник"] + [GEN_SRC[n] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["citation_rate", "citation_in_answer_rate"],
          [(LABELS[n - 1], [GEN[n].get("citation_rate"), GEN[n].get("citation_in_answer_rate")]) for n in (1, 2, 3)],
          "Рисунок 5 — Цитирование эталонного пункта", "%", ymax=100)
bar_chart(["groundedness_avg", "relevance_avg"],
          [(LABELS[n - 1], [GEN[n].get("groundedness_avg"), GEN[n].get("relevance_avg")]) for n in (1, 2, 3)],
          "Рисунок 6 — Groundedness и relevance", "0–1", ymax=1.05, fmt="{:.3f}")

| Метрика | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| citation_rate | 81,3 | 83,6 | 88,8 |
| citation_in_answer_rate | 69,8 | 72,0 | 74,6 |
| citation_in_context_rate | 81,3 | 83,6 | 88,8 |
| groundedness_avg | 0,893 | 0,908 | 0,908 |
| relevance_avg | 0,794 | 0,810 | 0,839 |
| Среднее время запроса, с | 6,35 | 6,91 | 10,15 |
| Ошибок live | 0 | 0 | 0 |
| Источник | stage4_gen_eval_results.json (04.11.2026 15:12) | stage4_gen_eval_results_e5.json (25.10.2026 21:36) | stage4_gen_eval_results_rerank.json (26.10.2026 01:26) |

### 5.7. Время, производительность и LLM-as-a-Judge

In [10]:
def durs(d):
    return [x.get("duration_sec") for x in (d or {}).get("details", []) if x.get("duration_sec") is not None]
TIME = {}
for n in (1, 2, 3):
    xs = durs(DATA[n]["rank"])
    TIME[n] = {"rag": GEN[n].get("duration_avg_sec"),
               "retr": sum(xs) / len(xs) if xs else None,
               "retr_med": float(np.median(xs)) if xs else None,
               "first": xs[0] if xs else None}
md_table(["Время"] + LABELS, [
    ["Полный RAG, с/вопрос (generation live, среднее)"] + [num(TIME[n]["rag"], 2) for n in (1, 2, 3)],
    ["Только поиск, с/вопрос (rank-прогон, среднее)"] + [num(TIME[n]["retr"], 3) for n in (1, 2, 3)],
    ["Только поиск, с/вопрос (медиана)"] + [num(TIME[n]["retr_med"], 3) for n in (1, 2, 3)],
    ["Первый запрос (холодный старт), с"] + [num(TIME[n]["first"], 2) for n in (1, 2, 3)],
])
bar_chart(["Полный RAG", "Только поиск"], [(LABELS[n - 1], [TIME[n]["rag"], TIME[n]["retr"]]) for n in (1, 2, 3)],
          "Рисунок 7 — Среднее время на вопрос, с", "с", fmt="{:.2f}")

PK = [("ttft_avg_sec", "TTFT avg, с"), ("latency_avg_sec", "Latency avg, с"), ("latency_p95_sec", "Latency p95, с"),
      ("latency_p99_sec", "Latency p99, с"), ("tps_avg", "TPS, ток/с")]
rows = [[name] + [num(DATA[n]["perf"]["metrics"][k], 2) for n in (1, 2, 3)] for k, name in PK]
rows += [["RPS"] + [num(DATA[n]["perf"]["rps"], 2) for n in (1, 2, 3)],
         ["Доступность"] + [f'{num(DATA[n]["perf"]["availability_pct"])} % ({DATA[n]["perf"]["successful_requests"]}/{DATA[n]["perf"]["total_requests"]})' for n in (1, 2, 3)]]
md_table(["Нагрузочный тест Ollama"] + LABELS, rows)
bar_chart(["TTFT", "Latency avg", "Latency p95", "Latency p99"],
          [(LABELS[n - 1], [DATA[n]["perf"]["metrics"][k] for k, _ in PK[:4]]) for n in (1, 2, 3)],
          "Рисунок 8 — Нагрузочный тест Ollama, с", "с", fmt="{:.1f}")

JK = [("faithfulness_avg", "Faithfulness"), ("answer_relevancy_avg", "Answer relevancy"), ("context_recall_avg", "Context recall")]
JFAIL = {n: [d["id"] for d in DATA[n]["judge"]["details"] if "Ошибка вызова" in str((d.get("scores") or {}).get("reasoning", ""))] for n in (1, 2, 3)}
rows = [[name] + [num(DATA[n]["judge"]["summary_metrics"][k], 4) for n in (1, 2, 3)] for k, name in JK]
rows.append(["Оценено без сбоя"] + [f'{DATA[n]["judge"]["sample_count"] - len(JFAIL[n])}/{DATA[n]["judge"]["sample_count"]}'
                                     + (f" (сбой id {JFAIL[n]})" if JFAIL[n] else "") for n in (1, 2, 3)])
md_table(["LLM-as-a-Judge"] + LABELS, rows)
bar_chart([name for _, name in JK], [(LABELS[n - 1], [DATA[n]["judge"]["summary_metrics"][k] for k, _ in JK]) for n in (1, 2, 3)],
          "Рисунок 9 — LLM-as-a-Judge", "0–1", ymax=0.7, fmt="{:.3f}")

| Время | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| Полный RAG, с/вопрос (generation live, среднее) | 6,35 | 6,91 | 10,15 |
| Только поиск, с/вопрос (rank-прогон, среднее) | 0,049 | 0,049 | 0,929 |
| Только поиск, с/вопрос (медиана) | 0,042 | 0,042 | 0,694 |
| Первый запрос (холодный старт), с | 1,53 | 1,27 | 12,87 |

| Нагрузочный тест Ollama | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| TTFT avg, с | 18,72 | 18,31 | 18,73 |
| Latency avg, с | 27,81 | 27,20 | 27,83 |
| Latency p95, с | 39,58 | 39,93 | 41,51 |
| Latency p99, с | 47,62 | 43,86 | 50,48 |
| TPS, ток/с | 38,19 | 38,96 | 38,77 |
| RPS | 0,11 | 0,11 | 0,11 |
| Доступность | 100,0 % (268/268) | 100,0 % (268/268) | 100,0 % (268/268) |

| LLM-as-a-Judge | Опыт 1 (база) | Опыт 2 (E5) | Опыт 3 (реранкер) |
|---|---|---|---|
| Faithfulness | 0,4347 | 0,4459 | 0,4396 |
| Answer relevancy | 0,1847 | 0,1802 | 0,1888 |
| Context recall | 0,5060 | 0,4735 | 0,4933 |
| Оценено без сбоя | 268/268 | 268/268 | 267/268 (сбой id [178]) |

Нагрузочный тест не использует поиск, поэтому различия между опытами в нём — разброс очереди GPU. Судье подаётся одинаковый шаблонный контекст, поэтому его оценки почти не различаются.

## 6. Ноутбук или .py файл (прототип)

Этот ноутбук (`Этапы/Reports/etap4/Etap4_final.ipynb`) — итоговый отчёт этапа: только чтение результатов, вывод появляется после запуска ячеек. Оценку выполняют скрипты `scripts/stage4_eval/`:

| Файл | Назначение |
|---|---|
| `retrieval/eval_retrieval_hitk.py`, `_e5.py`, `_rerank.py` | Hit@k, опыты 1 / 2 / 3 |
| `retrieval/mrr_ndcg/eval_retrieval_rank.py` | MRR, MRR@k, nDCG@k, Hit@10 (`--mode base/e5/rerank`) |
| `retrieval/mrr_ndcg/rank_metrics.py` | сводная таблица `rank_metrics_comparison.md/.csv` |
| `generation/eval_rag_generation.py` | генерация live: citation, groundedness, relevance, время |
| `generation/eval_rag_performance.py` | нагрузочный тест Ollama |
| `generation/eval_rag_llm_judge.py` | LLM-as-a-Judge |
| `reports/build_stage4_summary.py` | сводка всех метрик трёх опытов (md/csv/json) |
| `demo/demo_stage4_milvus_search.py` | демо поиска в Milvus |
| `run_all.ps1`, `run_all_e5.ps1`, `run_all_rerank.ps1` | полный прогон опыта 1 / 2 / 3 |
| `run_rank_all.ps1` | MRR/nDCG для трёх опытов |

Ячейка ниже проверяет наличие скриптов. Следующая — повторный прогон (Docker, Milvus, Ollama, Windows/PowerShell); выполняется только при `RUN_LIVE = True` (ячейка раздела 0) и перезаписывает файлы результатов.

In [11]:
SCRIPTS = ["retrieval/eval_retrieval_hitk.py", "retrieval/eval_retrieval_hitk_e5.py", "retrieval/eval_retrieval_hitk_rerank.py",
           "retrieval/mrr_ndcg/eval_retrieval_rank.py", "retrieval/mrr_ndcg/rank_metrics.py",
           "generation/eval_rag_generation.py", "generation/eval_rag_performance.py", "generation/eval_rag_llm_judge.py",
           "reports/build_stage4_summary.py", "demo/demo_stage4_milvus_search.py",
           "run_all.ps1", "run_all_e5.ps1", "run_all_rerank.ps1", "run_rank_all.ps1"]
md_table(["Скрипт", "Есть"], [[f"`scripts/stage4_eval/{s}`", "да" if (ROOT / "scripts/stage4_eval" / s).is_file() else "НЕТ"] for s in SCRIPTS])
boot = ROOT / "scripts" / "stage4_eval" / "lib" / "_bootstrap.py"
paths = ROOT / "scripts" / "lib" / "report_paths.py"
print("scripts/stage4_eval/lib/_bootstrap.py:", "есть" if boot.is_file() else "НЕТ")
print("scripts/lib/report_paths.py:", "есть" if paths.is_file() else "НЕТ")

| Скрипт | Есть |
|---|---|
| `scripts/stage4_eval/retrieval/eval_retrieval_hitk.py` | да |
| `scripts/stage4_eval/retrieval/eval_retrieval_hitk_e5.py` | да |
| `scripts/stage4_eval/retrieval/eval_retrieval_hitk_rerank.py` | да |
| `scripts/stage4_eval/retrieval/mrr_ndcg/eval_retrieval_rank.py` | да |
| `scripts/stage4_eval/retrieval/mrr_ndcg/rank_metrics.py` | да |
| `scripts/stage4_eval/generation/eval_rag_generation.py` | да |
| `scripts/stage4_eval/generation/eval_rag_performance.py` | да |
| `scripts/stage4_eval/generation/eval_rag_llm_judge.py` | да |
| `scripts/stage4_eval/reports/build_stage4_summary.py` | да |
| `scripts/stage4_eval/demo/demo_stage4_milvus_search.py` | да |
| `scripts/stage4_eval/run_all.ps1` | да |
| `scripts/stage4_eval/run_all_e5.ps1` | да |
| `scripts/stage4_eval/run_all_rerank.ps1` | да |
| `scripts/stage4_eval/run_rank_all.ps1` | да |

scripts/stage4_eval/lib/_bootstrap.py: есть
scripts/lib/report_paths.py: есть


In [12]:
# ПОВТОРНЫЙ ПРОГОН ОЦЕНКИ — только при RUN_LIVE = True (нужны Docker, Milvus, Ollama, scripts/stage4_eval/lib/_bootstrap.py).
import subprocess
if not RUN_LIVE:
    print("RUN_LIVE = False — повторный прогон пропущен, используются сохранённые результаты.")
else:
    steps = [
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_e5.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_rerank.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_rank_all.ps1", "-SkipInfra"],
    ]
    for cmd in steps:
        print(">", " ".join(cmd))
        rc = subprocess.run(cmd, cwd=str(ROOT)).returncode
        if rc != 0:
            raise RuntimeError(f"Шаг завершился с кодом {rc}")
    print("Готово. Перезапустите ноутбук (Run All), чтобы перечитать результаты.")

RUN_LIVE = False — повторный прогон пропущен, используются сохранённые результаты.


## 7. Выводы

In [13]:
text = """
1. Прототип рабочего контура RAG по ПУЭ собран и протестирован на едином реестре из 268 вопросов: эмбеддинг запроса, поиск фрагментов в Milvus, генерация ответа Ollama на основе найденных фрагментов. Нагрузочное тестирование подтвердило 100% доступность LLM и полное отсутствие ошибок live-генерации во всех трёх опытах, что доказывает сквозную работоспособность контура от ввода запроса до формирования ответа на всём объёме тестовых данных.
2. Ранние прогоны опыта 1 на наборах разного размера и состава (от 25 до 300 вопросов) показали вариативность Hit@5 от 28,0% до 81,3%, обусловленную сменой списка вопросов. Сравнение трёх опытов ниже выполнено строго на одном фиксированном реестре из 268 вопросов, что гарантирует корректность сравнения метрик и исключает влияние состава выборки на результаты.
3. Опыт 1 (базовый поиск без префиксов E5 и без реранкера): Hit@1 составляет 58,2% (156 из 268), Hit@5 — 81,3% (218 из 268). В 156 случаях эталонный пункт занимает первую позицию среди фрагментов, передаваемых в промпт. Ещё в 62 случаях он присутствует в топ-5, но ниже первого места. В оставшихся 50 вопросах его нет в топ-5, и модель формирует ответ на основе нерелевантных фрагментов. Для нормативной документации это критично: при попадании в контекст нескольких соседних пунктов фрагмент, стоящий выше, оказывает доминирующее влияние на формулировку ответа.
4. Опыт 2 добавляет к той же модели эмбеддингов префиксы passage: и query:. Hit@1 составляет 60,1% (161 из 268), Hit@5 — 83,6% (224 из 268), что даёт прирост в 5 вопросов на первом месте и 6 вопросов в пятёрке. Время поиска осталось на уровне ~0,05 с, полное время запроса выросло с 6,35 с до 6,91 с. MRR изменился с 0,681 до 0,696. Префиксы незначительно корректируют вектор запроса; данная конфигурация была принята за основу для проверки эффективности реранкера.
5. Опыт 3 сохраняет префиксы и модифицирует механизм выдачи: Milvus возвращает 20 кандидатов, после чего bge-reranker-v2-m3 отбирает из них 5 лучших. Hit@1 достигает 72,0% (193 из 268), Hit@5 — 88,8% (238 из 268). По сравнению с опытом 2 это прирост в 32 вопроса на первом месте и чистый прирост в 14 вопросов в пятёрке (19 прежних промахов устранено, 5 новых появилось). В разделе 5 Hit@5 незначительно снизился с 97,4% до 92,3%. Показатели цитирования (88,8%) и relevance (0,839) не являются независимым преимуществом: первый полностью совпадает с Hit@5, второй рассчитывается по той же логике («пункт назван в ответе» / «пункт есть в контексте»). Полное время запроса составило 10,15 с против 6,91 с. Из дополнительных 3,24 с на работу реранкера приходится ~0,9 с (среднее 0,93 с, медиана 0,69 с против 0,05 с в опыте 2); остальная задержка распределена между этапами генерации и сетевым взаимодействием. Данная конфигурация целесообразна, если приоритетом является вывод верного пункта на первую позицию, а время ответа около 10 с является приемлемым компромиссом.
6. Ранговые метрики на глубине 10 характеризуют именно позицию эталонного пункта, а не факт его попадания в топ-5. MRR: 0,681 → 0,696 → 0,793; nDCG@5: 0,708 → 0,726 → 0,812 (опыты 1 → 2 → 3). Значение MRR 0,793 соответствует среднему рангу ~1,3, тогда как 0,696 — рангу ~1,4. Основной качественный сдвиг обеспечивает опыт 3, что коррелирует с ростом Hit@1 на 11,9 п.п., в то время как Hit@5 за тот же шаг вырос лишь на 5,2 п.п. Это подтверждает, что реранкер преимущественно оптимизирует порядок выдачи, поднимая нужный пункт на первое место. Часть прироста Hit@5 обусловлена расширением пула кандидатов до 20 перед финальным отбором: внутри фиксированной пятёрки реранкер меняет порядок, но не создает новые попадания.
7. Генерация текста ответа не полностью реализует потенциал найденного поиском контекста. Показатель citation_rate совпадает с Hit@5 во всех трёх опытах (81,3% / 83,6% / 88,8%), так как методика расчёта фиксирует лишь наличие номера пункта в извлеченном фрагменте. Лексическая опора на контекст (groundedness) в опытах 2 и 3 идентична и составляет 0,908. При этом номер пункта явно фигурирует в тексте ответа лишь в 69,8%, 72,0% и 74,6% случаев соответственно. В опыте 3 наблюдается разрыв в ~14 п.п. между фактом наличия пункта в контексте (88,8%) и его явным упоминанием в ответе (74,6%): фрагмент передан модели, но в итоговой формулировке номер не указан. Оценки моделью (LLM-judge) по опытам демонстрируют незначительные расхождения и не опираются на фактические ответы прогона, поэтому выбор конфигурации на их основе не производился.
8. Для дальнейшей разработки утверждена конфигурация опыта 3, демонстрирующая наилучшие результаты по Hit@1 и MRR при известной и приемлемой временнóй стоимости. Дальнейшая оптимизация имеет два вектора. По направлению поиска: анализ 30 промахов Hit@5 в опыте 3 (из которых 23 не найдены ни в одном из опытов), включая аудит разметки и логики нарезки (чанкинга) пунктов. По направлению генерации: доработка промпта для гарантированного явного указания номера пункта в ответе, а также перевод оценки моделью (LLM-judge) на фактические ответы и контекст конкретного прогона. Увеличение показателя Hit@5 само по себе не решает задачу повышения доли явных ссылок в тексте с текущих 74,6%.
"""
display(Markdown(text))


1. Прототип рабочего контура RAG по ПУЭ собран и протестирован на едином реестре из 268 вопросов: эмбеддинг запроса, поиск фрагментов в Milvus, генерация ответа Ollama на основе найденных фрагментов. Нагрузочное тестирование подтвердило 100% доступность LLM и полное отсутствие ошибок live-генерации во всех трёх опытах, что доказывает сквозную работоспособность контура от ввода запроса до формирования ответа на всём объёме тестовых данных.
2. Ранние прогоны опыта 1 на наборах разного размера и состава (от 25 до 300 вопросов) показали вариативность Hit@5 от 28,0% до 81,3%, обусловленную сменой списка вопросов. Сравнение трёх опытов ниже выполнено строго на одном фиксированном реестре из 268 вопросов, что гарантирует корректность сравнения метрик и исключает влияние состава выборки на результаты.
3. Опыт 1 (базовый поиск без префиксов E5 и без реранкера): Hit@1 составляет 58,2% (156 из 268), Hit@5 — 81,3% (218 из 268). В 156 случаях эталонный пункт занимает первую позицию среди фрагментов, передаваемых в промпт. Ещё в 62 случаях он присутствует в топ-5, но ниже первого места. В оставшихся 50 вопросах его нет в топ-5, и модель формирует ответ на основе нерелевантных фрагментов. Для нормативной документации это критично: при попадании в контекст нескольких соседних пунктов фрагмент, стоящий выше, оказывает доминирующее влияние на формулировку ответа.
4. Опыт 2 добавляет к той же модели эмбеддингов префиксы passage: и query:. Hit@1 составляет 60,1% (161 из 268), Hit@5 — 83,6% (224 из 268), что даёт прирост в 5 вопросов на первом месте и 6 вопросов в пятёрке. Время поиска осталось на уровне ~0,05 с, полное время запроса выросло с 6,35 с до 6,91 с. MRR изменился с 0,681 до 0,696. Префиксы незначительно корректируют вектор запроса; данная конфигурация была принята за основу для проверки эффективности реранкера.
5. Опыт 3 сохраняет префиксы и модифицирует механизм выдачи: Milvus возвращает 20 кандидатов, после чего bge-reranker-v2-m3 отбирает из них 5 лучших. Hit@1 достигает 72,0% (193 из 268), Hit@5 — 88,8% (238 из 268). По сравнению с опытом 2 это прирост в 32 вопроса на первом месте и чистый прирост в 14 вопросов в пятёрке (19 прежних промахов устранено, 5 новых появилось). В разделе 5 Hit@5 незначительно снизился с 97,4% до 92,3%. Показатели цитирования (88,8%) и relevance (0,839) не являются независимым преимуществом: первый полностью совпадает с Hit@5, второй рассчитывается по той же логике («пункт назван в ответе» / «пункт есть в контексте»). Полное время запроса составило 10,15 с против 6,91 с. Из дополнительных 3,24 с на работу реранкера приходится ~0,9 с (среднее 0,93 с, медиана 0,69 с против 0,05 с в опыте 2); остальная задержка распределена между этапами генерации и сетевым взаимодействием. Данная конфигурация целесообразна, если приоритетом является вывод верного пункта на первую позицию, а время ответа около 10 с является приемлемым компромиссом.
6. Ранговые метрики на глубине 10 характеризуют именно позицию эталонного пункта, а не факт его попадания в топ-5. MRR: 0,681 → 0,696 → 0,793; nDCG@5: 0,708 → 0,726 → 0,812 (опыты 1 → 2 → 3). Значение MRR 0,793 соответствует среднему рангу ~1,3, тогда как 0,696 — рангу ~1,4. Основной качественный сдвиг обеспечивает опыт 3, что коррелирует с ростом Hit@1 на 11,9 п.п., в то время как Hit@5 за тот же шаг вырос лишь на 5,2 п.п. Это подтверждает, что реранкер преимущественно оптимизирует порядок выдачи, поднимая нужный пункт на первое место. Часть прироста Hit@5 обусловлена расширением пула кандидатов до 20 перед финальным отбором: внутри фиксированной пятёрки реранкер меняет порядок, но не создает новые попадания.
7. Генерация текста ответа не полностью реализует потенциал найденного поиском контекста. Показатель citation_rate совпадает с Hit@5 во всех трёх опытах (81,3% / 83,6% / 88,8%), так как методика расчёта фиксирует лишь наличие номера пункта в извлеченном фрагменте. Лексическая опора на контекст (groundedness) в опытах 2 и 3 идентична и составляет 0,908. При этом номер пункта явно фигурирует в тексте ответа лишь в 69,8%, 72,0% и 74,6% случаев соответственно. В опыте 3 наблюдается разрыв в ~14 п.п. между фактом наличия пункта в контексте (88,8%) и его явным упоминанием в ответе (74,6%): фрагмент передан модели, но в итоговой формулировке номер не указан. Оценки моделью (LLM-judge) по опытам демонстрируют незначительные расхождения и не опираются на фактические ответы прогона, поэтому выбор конфигурации на их основе не производился.
8. Для дальнейшей разработки утверждена конфигурация опыта 3, демонстрирующая наилучшие результаты по Hit@1 и MRR при известной и приемлемой временнóй стоимости. Дальнейшая оптимизация имеет два вектора. По направлению поиска: анализ 30 промахов Hit@5 в опыте 3 (из которых 23 не найдены ни в одном из опытов), включая аудит разметки и логики нарезки (чанкинга) пунктов. По направлению генерации: доработка промпта для гарантированного явного указания номера пункта в ответе, а также перевод оценки моделью (LLM-judge) на фактические ответы и контекст конкретного прогона. Увеличение показателя Hit@5 само по себе не решает задачу повышения доли явных ссылок в тексте с текущих 74,6%.


## 8. План дальнейшей работы

К моменту этого отчёта перечисленные работы ещё впереди.

1. Подключить RAG (конфигурация опыта 3) к Telegram-боту (`src/bot/`).
2. Разобрать промахи опыта 3 и общие промахи трёх опытов; исправить разметку набора вопросов.
3. Доработать промпт, чтобы LLM явно указывал номер пункта в ответе.
4. Перевести оценку моделью (LLM-judge) на фактические ответы и контекст из прогона генерации.
5. Измерить задержку полного контура RAG; собрать бот, Milvus и Ollama в один Docker Compose.
6. По возможности добавить ПТЭЭП.

## Приложения

**Соответствие нумерации:** опыт 1 = опыт 0 в файлах проекта, опыт 2 = опыт 1 (E5), опыт 3 = опыт 2 (реранкер).

| Что | Файл |
|---|---|
| Набор вопросов (268) | `data/rag_eval_questions/eval_questions.json` |
| Hit@k | `scripts/stage4_eval/results/retrieval/stage4_hitk_results{,_e5,_rerank}.json` |
| История прогонов опыта 1 | `scripts/stage4_eval/results/retrieval/stage4_hitk_runs.jsonl` |
| MRR/nDCG | `scripts/stage4_eval/results/retrieval/stage4_rank_results_{base,e5,rerank}.json` |
| Генерация, performance, judge | `scripts/stage4_eval/results/generation/` (без суффикса, `_e5`, `_rerank`) |
| Паспорта прогонов | `Этапы/Reports/etap4/Etap4-hitk-otchet-N268.md`, `Etap4-opyt1-e5-N268.md`, `Etap4-opyt2-rerank-N268.md` |

## Приложения

**Соответствие нумерации:** опыт 1 = опыт 0 в файлах проекта, опыт 2 = опыт 1 (E5), опыт 3 = опыт 2 (реранкер).

| Что | Файл |
|---|---|
| Набор вопросов (268) | `data/rag_eval_questions/eval_questions.json` |
| Hit@k | `scripts/stage4_eval/results/retrieval/stage4_hitk_results{,_e5,_rerank}.json` |
| История прогонов опыта 1 | `scripts/stage4_eval/results/retrieval/stage4_hitk_runs.jsonl` |
| MRR/nDCG | `scripts/stage4_eval/results/retrieval/stage4_rank_results_{base,e5,rerank}.json` |
| Генерация, performance, judge | `scripts/stage4_eval/results/generation/` (без суффикса, `_e5`, `_rerank`) |
| Паспорта прогонов | `Этапы/Reports/etap4/Etap4-hitk-otchet-N268.md`, `Etap4-opyt1-e5-N268.md`, `Etap4-opyt2-rerank-N268.md` |